# Perform name conversions for MetaboAnalyst analyses
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from itertools import product

import pandas as pd

from hk1_r12ter_analysis.config import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    logger,
)
from hk1_r12ter_analysis.io import (
    copy_file_to_destination,
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load data

In [ ]:
sample_key = "Sample"
group_key = "Group"

source_types = ["RBC", "Plasma"]
data_types = ["Metabolites", "Lipids", "Oxylipins", "CombinedLipids"]
value_type = "Intensities"

normalization_inputs = (
    # Sample Normalization, Data Transformation, and Data Scaling
    [None, None, None],
    ["median", None, None],
    ["median", None, "auto"],
)

# Input data arguments
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized"
output_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "MetaboAnalyst" / "Normalized"
output_dirpath.mkdir(parents=True, exist_ok=True)

### ID corrections for enrichment and pathway analyses

In [ ]:
old_name_col = "Omics Name"
new_name_col = "MetaboAnalyst Name"
mapping_dict = {}

data_types = ["Metabolites", "Lipids", "Oxylipins"]
for data_type in data_types:
    idmap_filename = RAW_DATA_DIR / "HK1-R12Ter" / make_filename(data_type, "Metadata")
    df_map = pd.read_csv(idmap_filename)
    df_map = df_map.dropna(subset=new_name_col, axis=0)
    # Map MetaboAnalyst names to those in the omics data
    mapping_dict.update(df_map.set_index(old_name_col)[new_name_col].to_dict())

### Load Intensities

In [ ]:
normalization_strs = [
    "-".join([str(val).lower() for val in values]) for values in normalization_inputs
]
index_cols = [key for key in [sample_key, group_key] if key]
for norm_str in normalization_strs:
    input_dirpath_norm = input_dirpath / norm_str
    output_dirpath_norm = output_dirpath / norm_str
    output_dirpath_norm.mkdir(parents=True, exist_ok=True)

    for source_type, data_type in product(source_types, data_types):
        logger.debug(f"Processing '{source_type}-{data_type}' data...")
        filename = make_filename(source_type, data_type, value_type)
        df = load_dataframe_from_file(input_dirpath_norm / filename, index_col=index_cols)
        logger.debug(f"Converting IDs for '{source_type}-{data_type}'.")
        df = df.rename(mapping_dict, axis=1)

        df_not_mapped = df.loc[:, ~df.columns.isin(list(mapping_dict.values()))]
        logger.info(
            f"Number of '{source_type}-{data_type}' unmapped: {len(df_not_mapped.columns)}."
        )

        df = df.loc[:, df.columns.isin(list(mapping_dict.values()))]
        logger.info(f"Number of '{source_type}-{data_type}' mapped: {len(df.columns)}.")

        # Save updated data for MetaboAnalyst Analysis
        df = df.sort_index(level=[1, 0], ascending=[False, True])
        filename = make_filename(source_type, data_type, value_type)
        save_dataframe_to_file(df, output_dirpath_norm / filename, index=True)
    logger.info(f"Processed data for '{source_type}-{data_type}'.")

    # Proteins do not require ID conversions
    data_type = "Proteins"
    for source_type in source_types:
        filename = make_filename(source_type, data_type, value_type)
        copy_file_to_destination(
            source_file=input_dirpath_norm / filename,
            destination_path=output_dirpath_norm / filename,
        )